In [31]:
from loaders import CSVLoader

loader = CSVLoader("../../data for sharing_csv", ["P2","P3","P4","P5","P6","P7","P8"])

# Checkpoint 1: 验证发现 7 个地点
len(loader.location_files) #"应发现 7 个地点 CSV"


2026-04-06 20:53:37,335 - INFO - Discovered 7 location files: ['P2', 'P3', 'P4', 'P5', 'P6', 'P7', 'P8']


7

In [32]:

# Checkpoint 2: 加载单地点
df_p2 = loader.load_location("P2")
df_p2.columns


2026-04-06 20:53:39,010 - INFO - Loaded P2: 6524 records


Index(['GPS_Time(s)', 'PRN', 'nSV', 'pseudorange', 'C/N0', 'Elevation',
       'Azimuth', 'err_tropo', 'err_iono', 'sat_clock_error',
       'Pseudorange_residual', 'Normalized_Pseudorange_residual',
       'Pr_rate_consitency', 'Sat_pos_x', 'Sat_pos_y', 'Sat_pos_z',
       'LOS/NLOS_label', 'location'],
      dtype='str')

In [33]:

# Checkpoint 3: 合并所有地点
df_merged = loader.load_and_merge()
len(df_merged) 
df_merged["location"].nunique()

2026-04-06 20:53:41,565 - INFO - Loading P2.csv...
2026-04-06 20:53:41,585 - INFO - Loading P3.csv...
2026-04-06 20:53:41,617 - INFO - Loading P4.csv...
2026-04-06 20:53:41,725 - INFO - Loading P5.csv...
2026-04-06 20:53:41,792 - INFO - Loading P6.csv...
2026-04-06 20:53:41,822 - INFO - Loading P7.csv...
2026-04-06 20:53:41,859 - INFO - Loading P8.csv...
2026-04-06 20:53:41,913 - INFO - Merged dataset shape: (86830, 18)


7

In [ ]:
# Checkpoint 4: 过滤异常数据
from importlib import reload
import filters
reload(filters)
from filters import DataFilter

# 使用之前加载的 df_merged 真实数据
print("原始数据:")
print(f"形状：{df_merged.shape}")

filter_obj = DataFilter()
df_filtered = filter_obj.filter_outliers(df_merged)

print("\n过滤后数据:")
print(f"形状：{df_filtered.shape}")

2026-04-06 21:17:10,021 - INFO - Dropped rows with NaN: 86830 -> 86830 (removed 0)
2026-04-06 21:17:10,049 - INFO - Filtered Pr_rate_consitency (value == 9999.0): 86830 -> 85393 (removed 1437)
2026-04-06 21:17:10,051 - INFO - Total filtering: 86830 -> 85393 records (98.3%), removed 1437 outliers


原始数据:
形状：(86830, 18)

过滤后数据:
形状：(85393, 18)


In [ ]:
# Checkpoint 5: 标签映射
y = filter.map_labels(df_filtered)

7        0
8        1
9        1
10       1
11       1
        ..
86825    0
86826    0
86827    1
86828    0
86829    0
Name: LOS/NLOS_label, Length: 85393, dtype: int64

In [73]:
# Checkpoint 6: 测试 WindowGenerator
from importlib import reload
import windowers
reload(windowers)
from windowers import WindowGenerator
import numpy as np

# 使用过滤后的真实数据
print("输入数据形状:", df_filtered.shape)
print()

window_gen = WindowGenerator(window_size=10, max_satellites=30)

# 生成 STCA 双通道输入
X_temporal, X_spatial, y, locations = window_gen.generate_stca_inputs(df_filtered)

print("时间通道输入形状:", X_temporal.shape)  # (N, 10, 4)
print("空间通道输入形状:", X_spatial.shape)  # (N, 30, 4)
print("标签形状:", y.shape)
print("地点数组形状:", locations.shape)
print()
print("标签分布：NLOS =", np.sum(y==0), ", LOS =", np.sum(y==1))

2026-04-06 21:58:03,841 - INFO - Temporal input shape: (85177, 10, 4)


输入数据形状: (85393, 18)



2026-04-06 21:58:07,298 - INFO - Spatial input shape: (85177, 30, 4)


时间通道输入形状: (85177, 10, 4)
空间通道输入形状: (85177, 30, 4)
标签形状: (85177,)
地点数组形状: (85177,)

标签分布：NLOS = 42632 , LOS = 42545
